In [ ]:
# 패키지 설치
!pip install ftfy regex tqdm -q
!pip install git+https://github.com/openai/CLIP.git -q
!pip install faiss-cpu -q

In [ ]:
# CLIP 모델 로드
import clip
import torch
from PIL import Image
import numpy as np
import faiss

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print(f"CLIP 로드 완료 / 디바이스: {device}")

In [ ]:
# 이미지 임베딩 DB 구축
IMG_DIR = '/content/drive/MyDrive/SafeSight/data/raw/images'
img_files = [f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')]
print(f"총 이미지: {len(img_files)}장")

embeddings = []
valid_ids = []

for i, fname in enumerate(img_files):
    img_path = os.path.join(IMG_DIR, fname)
    try:
        image = preprocess(
            Image.open(img_path).convert("RGB")
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            emb = model.encode_image(image)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            embeddings.append(emb.cpu().numpy())
            valid_ids.append(fname.replace('.jpg', ''))

    except Exception as e:
        print(f"실패: {fname} → {e}")

    if (i+1) % 100 == 0:
        print(f"진행중: {i+1}/{len(img_files)}장")

# FAISS DB 저장
embeddings = np.vstack(embeddings).astype('float32')
index = faiss.IndexFlatIP(512)
index.add(embeddings)

os.makedirs('/content/drive/MyDrive/SafeSight/data/embeddings', exist_ok=True)
faiss.write_index(
    index,
    '/content/drive/MyDrive/SafeSight/data/embeddings/image_index.faiss'
)
pd.DataFrame({'desertionNo': valid_ids}).to_csv(
    '/content/drive/MyDrive/SafeSight/data/embeddings/index_map.csv',
    index=False
)
print(f"FAISS DB 구축 완료! 총 {len(valid_ids)}개")

In [ ]:
# 검색 함수
metadata = pd.read_csv(
    '/content/drive/MyDrive/SafeSight/data/raw/metadata.csv'
)
index_map = pd.read_csv(
    '/content/drive/MyDrive/SafeSight/data/embeddings/index_map.csv'
)
def search(query, k=5, animal_type=None):
    try:
        query_en = GoogleTranslator(source='ko', target='en').translate(query)
    except:
        query_en = query

    text = clip.tokenize([query_en]).to(device)
    with torch.no_grad():
        text_emb = model.encode_text(text)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
        text_emb = text_emb.cpu().numpy().astype('float32')

    # 전체 DB에서 검색 (1810장 전부!)
    total = index.ntotal
    similarities, indices = index.search(text_emb, total)

    CAT_KEYWORDS = ['한국 고양이', '묘', '코리안숏헤어', '페르시안', '러시안블루', '스핑크스', '랙돌', '샴']

    results = []
    for sim, idx in zip(similarities[0], indices[0]):
        desertion_no = index_map.iloc[idx]['desertionNo']
        meta_row = metadata[metadata['desertionNo'] == int(desertion_no)]
        if len(meta_row) == 0:
            continue
        meta = meta_row.iloc[0]

        # 동물 종류 필터
        if animal_type == "🐶 강아지":
            if any(cat in str(meta['kindNm']) for cat in CAT_KEYWORDS):
                continue
        if animal_type == "🐱 고양이":
            if not any(cat in str(meta['kindNm']) for cat in CAT_KEYWORDS):
                continue

        results.append({
            "id": str(desertion_no),
            "similarity": round(float(sim) * 100, 1),
            "kindNm": meta['kindNm'],
            "colorCd": meta['colorCd'],
            "specialMark": meta['specialMark'],
            "happenPlace": meta['happenPlace'],
            "orgNm": meta['orgNm'],
            "img_path": f"{IMG_DIR}/{desertion_no}.jpg"
        })

        if len(results) >= k:
            break

    return results
